# 03. Hierarchical observations and sampling

![Hierarchical groups and balanced estimation](../images/03_hierarchical_observations.svg)

**Learning goals:** compute spread and empirical distributions, simulate dependent groups, expose pseudoreplication, compare row and group estimands, sample conditionally, and create group-disjoint partitions.

In [ ]:
import random
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

SEED = 19
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
print(f'numpy={np.__version__}, synthetic data only')

## 1. Means imply weights

The row mean gives group $g$ weight $n_g/N$. The group-balanced mean first averages within each group and then gives every group weight $1/G$. Both can be correct, but they answer different population questions.

In [ ]:
values = np.array([0., 0., 10., 10., 10., 10., 10., 10., 10., 10.])
groups = np.array([0, 0] + [1] * 8)
row_mean = values.mean()
group_means = np.array([values[groups == g].mean() for g in np.unique(groups)])
balanced_mean = group_means.mean()
assert row_mean == 8.0 and balanced_mean == 5.0
print(f'row-weighted={row_mean:.1f}, group-balanced={balanced_mean:.1f}')

## 2. Variance, standard deviation, and quantiles

Population variance divides squared deviations by $N$. Sample variance uses $N-1$ (`ddof=1`) when estimating a broader population variance. Standard deviation restores the original units. Quantiles describe ranks and are less sensitive to extreme values than means.

In [ ]:
sample = np.array([1., 2., 2., 3., 100.])
population_var = np.var(sample, ddof=0)
sample_var = np.var(sample, ddof=1)
sample_std = np.std(sample, ddof=1)
quartiles = np.quantile(sample, [0.25, 0.5, 0.75], method='linear')
assert sample_var > population_var
assert np.isclose(sample_std ** 2, sample_var)
assert quartiles[1] == 2.0
print('sample variance/std:', round(sample_var, 2), round(sample_std, 2))
print('quartiles:', quartiles)

## 3. An empirical cumulative distribution

The ECDF at threshold $a$ is the observed fraction at or below $a$. Sorting once and using `np.searchsorted(..., side='right')` is efficient for many thresholds because it uses binary search instead of a full comparison per query.

In [ ]:
def ecdf(values, thresholds):
    sorted_values = np.sort(np.asarray(values))
    thresholds = np.asarray(thresholds)
    return np.searchsorted(sorted_values, thresholds, side='right') / len(sorted_values)

thresholds = np.array([1., 2., 3., 100.])
probabilities = ecdf(sample, thresholds)
np.testing.assert_allclose(probabilities, [0.2, 0.6, 0.8, 1.0])
assert np.all(np.diff(probabilities) >= 0)
print(list(zip(thresholds, probabilities)))

## 4. Simulate clustered dependence

Use $y_{gi}=\mu+a_g+e_{gi}$. All rows in group $g$ share the random effect $a_g$. The intraclass correlation is $\rho=\sigma_a^2/(\sigma_a^2+\sigma_e^2)$.

In [ ]:
G, rows_per_group = 40, 20
sigma_group, sigma_noise = 2.0, 1.0
group_effect = rng.normal(0, sigma_group, size=G)
group_id = np.repeat(np.arange(G), rows_per_group)
outcome = 5.0 + group_effect[group_id] + rng.normal(0, sigma_noise, size=G*rows_per_group)
rho_theory = sigma_group**2 / (sigma_group**2 + sigma_noise**2)
assert outcome.shape == group_id.shape == (800,)
assert np.unique(group_id).size == G
print(f'rows={len(outcome)}, groups={G}, theoretical ICC={rho_theory:.2f}')

## 5. Pseudoreplication changes uncertainty

With equal group size $m$, the design effect is approximately $1+(m-1)\rho$. Dividing row count by it gives a rough effective sample size. A naive row-level standard error ignores shared group effects. A group-level standard error respects the independent sampling units in this simulation.

In [ ]:
naive_se = outcome.std(ddof=1) / np.sqrt(len(outcome))
_, inverse = np.unique(group_id, return_inverse=True)
sums = np.bincount(inverse, weights=outcome)
counts = np.bincount(inverse)
observed_group_means = sums / counts
group_se = observed_group_means.std(ddof=1) / np.sqrt(G)
design_effect = 1 + (rows_per_group - 1) * rho_theory
effective_n = len(outcome) / design_effect
assert group_se > naive_se
print(f'naive row SE={naive_se:.3f}, group SE={group_se:.3f}, effective N~{effective_n:.1f}')

## 6. Conditional sampling changes group probabilities

Direct row sampling selects group $g$ with probability $n_g/N$. Two-stage sampling first chooses a group uniformly, then a row inside it, giving each group probability $1/G$. The code below makes that distinction concrete.

In [ ]:
unequal_groups = np.array([0]*2 + [1]*8)
row_draw_groups = unequal_groups[rng.integers(0, len(unequal_groups), size=20_000)]
two_stage_groups = rng.integers(0, 2, size=20_000)
row_p_group1 = np.mean(row_draw_groups == 1)
balanced_p_group1 = np.mean(two_stage_groups == 1)
assert abs(row_p_group1 - 0.8) < 0.02
assert abs(balanced_p_group1 - 0.5) < 0.02
print(f'P(group 1): row sampling={row_p_group1:.3f}, two-stage={balanced_p_group1:.3f}')

## 7. Split whole groups, not related rows

`GroupShuffleSplit` keeps every group's rows together. This prevents a model from seeing one recording from a participant during training and another from the same participant during testing. Always verify the realized group sets.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
train_idx, test_idx = next(splitter.split(outcome[:, None], groups=group_id))
train_groups = set(group_id[train_idx])
test_groups = set(group_id[test_idx])
assert train_groups.isdisjoint(test_groups)
assert len(train_idx) + len(test_idx) == len(outcome)
print(f'train groups={len(train_groups)}, test groups={len(test_groups)}, overlap=0')

## Exercises and final takeaways

**Exercises:** (1) Change the group and noise standard deviations and predict the ICC and design effect. (2) Create unequal group sizes and compare row, equal-group, and custom weighted means. (3) Repeat the group split several times and summarize test-group counts.

**Takeaways:** descriptive statistics encode weights; repeated rows are not automatically independent evidence; the experimental unit determines uncertainty and splitting; vectorized group summaries with `np.bincount` are both clear and efficient.

## Continue learning

[Previous notebook: 02](02_inner_product_geometry.ipynb) | [Lecture](../lectures/03_hierarchical_observations.md) | [Curriculum](../README.md) | [Next notebook: 04](04_attention_and_positions.ipynb)